# Whisper Large V3 Turbo: Baseline Analysis and Training

This notebook performs a complete workflow for the Cadenza challenge:
1.  **Setup**: Installs dependencies and downloads baseline scripts.
2.  **Data Preparation**: Downloads and extracts the Cadenza training and validation datasets.
3.  **Baseline Execution**: Runs predictions using both STOI and Whisper baselines.
4.  **Whisper Large V3 Analysis**: Computes scores and runs predictions for the `large-v3-turbo` model.
5.  **Custom Model Training**: Trains a regression head on top of frozen Whisper embeddings to predict intelligibility scores.
6.  **Submission**: Generates a `submission.csv` file based on the best-trained model.

## 1. Environment Setup and Data Download

This section installs necessary system packages, downloads the baseline code from the Clarity Challenge repository, and installs the required Python libraries.

In [ ]:
# Install aria2 for accelerated downloads and download the training data
!apt-get update && apt-get install -y aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://zenodo.org/records/17252365/files/cadenza_clip1_data.train.v1.0.tar.gz?download=1" -o cadenza_clip1_data.train.v1.0.tar.gz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 165 not upgraded.
Need to get 1,468 kB/1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libaria2-0 amd64 1.36.0-1 [1,086 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 aria2 amd64 1.36.0-1 [381 kB]
Fetched 786 kB in 2s (452 kB/s)

78Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 128639 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [  8%] [####......................................

In [ ]:
# Extract the training data, then remove the archive to save space
!tar -xzvf cadenza_clip1_data.train.v1.0.tar.gz
!rm cadenza_clip1_data.train.v1.0.tar.gz

# Verify the contents
print("Verifying extracted data structure")
!ls -l cadenza_data 
!ls -l cadenza_data/metadata
!ls -l cadenza_data/train

In [ ]:
# Create a directory for the baseline code
!mkdir -p baseline

# Download the official baseline scripts from the Clarity Challenge GitHub repository
!npx -y degit --force claritychallenge/clarity/recipes/cad_icassp_2026/baseline#main baseline

In [ ]:
import os
import sys

# Change the working directory to the baseline folder
os.chdir('baseline')
print(f"Current working directory: {os.getcwd()}")

# Install Python packages
print("\nInstalling required packages...")
# Install standard packages for the baseline
!pip install -q hydra-core omegaconf pandas pystoi demucs openai-whisper jiwer inflect

# Install the pyclarity library directly from its GitHub repository
print("\nInstalling pyclarity library from GitHub...")
!pip install -q git+https://github.com/claritychallenge/clarity.git

print("\nEnvironment setup complete.")

## 2. Baseline Preparation and Execution

Here, we prepare the environment for the baseline models by copying pre-computed scores and downloading the validation data. We then run the STOI and standard Whisper baselines to generate initial predictions.

In [ ]:
import os

# Create the experiment directory
exp_dir = "exp"
os.makedirs(exp_dir, exist_ok=True)

# Define the path to local precomputed scores
local_precomputed_path = "precomputed"

# Copy pre-computed scores into the experiment directory
print("Copying STOI and Whisper scores...")
!cp {local_precomputed_path}/cadenza_data.train.stoi.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.valid.stoi.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.train.whisper.jsonl {exp_dir}/
!cp {local_precomputed_path}/cadenza_data.valid.whisper.jsonl {exp_dir}/

print(f"\nPre-computed scores copied to the '{exp_dir}/' directory.")
!ls -l {exp_dir}

In [ ]:
import os

# Return to the main working directory to download validation data
os.chdir('/kaggle/working')

print("Downloading Validation Data")
!aria2c -c -x 16 -s 16 -k 1M -o cadenza_clip1_data.valid.v1.0.tar.gz "https://zenodo.org/records/17252365/files/cadenza_clip1_data.valid.v1.0.tar.gz"

print("\nExtracting Validation Data")
!tar -xzvf cadenza_clip1_data.valid.v1.0.tar.gz
!rm cadenza_clip1_data.valid.v1.0.tar.gz

print("\nValidation data is now in place.")
!ls -l cadenza_data/metadata

# Return to the baseline directory for subsequent steps
os.chdir('baseline')
print(f"\nReturned to directory: {os.getcwd()}")

In [ ]:
# The precomputed score files use a different key ("whisper.mixture") than the scripts expect ("whisper").
# This cell corrects the key in the JSONL files.

print("Inspecting first line of the whisper score file")
!head -n 1 exp/cadenza_data.train.whisper.jsonl

print('\nCorrecting the key in whisper.jsonl files')
!sed -i 's/"whisper.mixture"/"whisper"/g' exp/cadenza_data.train.whisper.jsonl
!sed -i 's/"whisper.mixture"/"whisper"/g' exp/cadenza_data.valid.whisper.jsonl

print("\nKeys have been corrected. Verifying the change:")
!head -n 1 exp/cadenza_data.train.whisper.jsonl

In [ ]:
# Run STOI baseline prediction
print("Generating predictions for STOI baseline")
!python predict.py baseline=stoi data.cadenza_data_root=/kaggle/working
print("\nSTOI prediction complete.")

# Run Whisper baseline prediction
print("\nGenerating predictions for Whisper baseline")
!python predict.py baseline=whisper data.cadenza_data_root=/kaggle/working
print("\nWhisper prediction complete.")

In [ ]:
print('Previewing STOI Predictions')
!head exp/cadenza_data.stoi.valid.predict.csv

print('\nPreviewing Whisper Predictions')
!head exp/cadenza_data.whisper.valid.predict.csv

## 3. Whisper `large-v3-turbo` Analysis

This section focuses on using the more advanced `large-v3-turbo` version of Whisper. We first set up the correct directory symbolic links, then compute the Whisper scores for this model on both the training and validation sets. Finally, we generate predictions and evaluate the results.

In [ ]:
%%bash
# The baseline scripts expect a specific 'audio' subdirectory.
# We create this directory and then create symbolic links to the actual data locations.
# This avoids duplicating data and ensures the scripts can find the audio files.

# Create the audio directory if it doesn't exist
mkdir -p /kaggle/working/cadenza_data/audio

# Remove any existing broken symlinks (ignore errors if they don't exist)
rm -f /kaggle/working/cadenza_data/audio/train
rm -f /kaggle/working/cadenza_data/audio/valid

# Create correct symlinks pointing to the actual data locations
# Note: Using consistent /kaggle/working/ path
ln -s /kaggle/working/cadenza_data/train /kaggle/working/cadenza_data/audio/train
ln -s /kaggle/working/cadenza_data/valid /kaggle/working/cadenza_data/audio/valid

# Verify the new symlinks
echo "Verifying new symlinks:"
ls -la /kaggle/working/cadenza_data/audio/
echo ""
echo "Checking if we can access the files:"
ls /kaggle/working/cadenza_data/audio/train/signals/ | head -5

In [ ]:
# Compute Whisper scores for the TRAINING set using the large-v3-turbo model.
# The `cd` command ensures the script is run from the correct directory.
!cd /kaggle/working/baseline/ && python compute_whisper.py \
  split=train \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.whisper_version=large-v3 \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# Compute Whisper scores for the VALIDATION set using the large-v3-turbo model.
!cd /kaggle/working/baseline/ && python compute_whisper.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.whisper_version=large-v3 \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# Verify that the new score files have been created in the experiment directory.
print("Checking for new whisper-large-v3-turbo score files")
!ls -lh /kaggle/working/baseline/exp/

In [ ]:
# Generate predictions on the validation set using the newly computed scores.
!cd /kaggle/working/baseline/ && python predict.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.system=whisper-large-v3-turbo

In [ ]:
# Verify that the new prediction file has been created.
print("Checking for new prediction file")
!ls -lh /kaggle/working/baseline/exp/

In [ ]:
# Run the evaluation script to get RMSE and correlation metrics for the new predictions.
# Note: Corrected data path from /content/ to /kaggle/working/ for consistency.
!cd /kaggle/working/baseline/ && python evaluate.py \
  split=valid \
  data.cadenza_data_root=/kaggle/working/ \
  baseline=whisper \
  baseline.system=whisper-large-v3-turbo

## 4. Custom Model Training

This section defines and trains a custom regression model. The model uses the frozen encoder from Whisper `large-v3-turbo` to generate powerful audio embeddings. A small regression head is then trained on top of these embeddings to predict the final intelligibility score.

In [ ]:
# CRITICAL FIX: Install a specific version of protobuf to prevent model loading errors
# with the transformers library. Also install other necessary libraries for training.
!pip install -q "transformers==4.34.0" datasets accelerate torch pandas librosa soundfile "protobuf==3.20.3" sentencepiece

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,452 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,827 kB]
Fetched 12.5 MB in 5s (2,567 kB/s)          

In [ ]:
# Download and Prepare Full Dataset

import os

# Configuration
WORKING_DIR = "/kaggle/working"
DATA_DIR = os.path.join(WORKING_DIR, "cadenza_data_full")
TRAIN_URL = "https://zenodo.org/records/17252365/files/cadenza_clip1_data.train.v1.0.tar.gz?download=1"
VALID_URL = "https://zenodo.org/records/17252365/files/cadenza_clip1_data.valid.v1.0.tar.gz?download=1"
TRAIN_ARCHIVE = os.path.join(WORKING_DIR, "cadenza_clip1_data.train.v1.0.tar.gz")
VALID_ARCHIVE = os.path.join(WORKING_DIR, "cadenza_clip1_data.valid.v1.0.tar.gz")

# Download
print("Downloading training data...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{TRAIN_URL}" -o "{TRAIN_ARCHIVE}"
print("Training data downloaded.")

print("\nDownloading validation data...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{VALID_URL}" -o "{VALID_ARCHIVE}"
print("Validation data downloaded.")

# Extract
os.makedirs(DATA_DIR, exist_ok=True)
print("\nExtracting data...")
# Extract and strip the top-level directory to maintain a clean structure
!tar -xzf {TRAIN_ARCHIVE} --strip-components=1 -C {DATA_DIR}
!tar -xzf {VALID_ARCHIVE} --strip-components=1 -C {DATA_DIR}
print("Data extracted.")

# Cleanup
print("\nCleaning up archives...")
!rm {TRAIN_ARCHIVE} {VALID_ARCHIVE}
print("Cleanup complete.")

print("\nFinal directory structure:")
!ls -l {DATA_DIR}

 *** Download Progress Summary as of Sat Nov 15 10:49:44 2025 ***              3m43s]m0m
[#04dbe3 664MiB/4.2GiB(15%) CN:16 DL:12MiB ETA:5m]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-

 *** Download Progress Summary as of Sat Nov 15 10:50:45 2025 ***              3m33s]
[#04dbe3 1.6GiB/4.2GiB(38%) CN:16 DL:14MiB ETA:2m57s]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-

 *** Download Progress Summary as of Sat Nov 15 10:51:46 2025 ***              1m27s]
[#04dbe3 2.6GiB/4.2GiB(62%) CN:16 DL:18MiB ETA:1m27s]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-

 *** Download Progress Summary as of Sat Nov 15 10:52:47 2025 ***              33s]mm
[#04dbe3 3.6GiB/4.2GiB(87%) CN:16 DL:16MiB ETA:32s]
FILE: /kaggle/working/cadenza_clip1_data.train.v1.0.tar.gz
-

[#04dbe3 4.2GiB/4.2GiB(99%) CN:3 DL:9.2MiB]0m]m
Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
04dbe3|OK  |   

In [ ]:
# Load and Prepare Dataset

from datasets import load_dataset, DatasetDict
import os

# Define paths based on the new extracted data location
metadata_path = "/kaggle/working/cadenza_data_full/metadata/train_metadata.json"
audio_base_path = "/kaggle/working/cadenza_data_full/train/signals"

# Load the metadata into a dataset
full_dataset = load_dataset('json', data_files=metadata_path, split='train')

# Define a function to add the full audio file path to each example
def add_audio_path(example):
    example['audio_path'] = os.path.join(audio_base_path, f"{example['signal']}.flac")
    return example

# Apply the function and split the data into training and validation sets (90/10 split)
full_dataset = full_dataset.map(add_audio_path)
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
split_dataset["validation"] = split_dataset.pop("test")

print("Dataset prepared and split:")
print(split_dataset)

Dataset prepared and split:
DatasetDict({
    train: Dataset({
        features: ['signal', 'prompt', 'response', 'n_words', 'words_correct', 'correctness', 'hearing_loss', 'audio_path'],
        num_rows: 7921
    })
    validation: Dataset({
        features: ['signal', 'prompt', 'response', 'n_words', 'words_correct', 'correctness', 'hearing_loss', 'audio_path'],
        num_rows: 881
    })
})


In [ ]:
# Define the Custom PyTorch Dataset

import torch
from torch.utils.data import Dataset
import librosa

class CadenzaDataset(Dataset):
    """
    Custom PyTorch dataset to load audio, process it for the Whisper model,
    and return the necessary tensors for training the regression head.
    """
    def __init__(self, hf_dataset, processor):
        self.hf_dataset = hf_dataset
        self.processor = processor
        self.hearing_loss_map = {"No Loss": 0, "Mild": 1, "Moderate": 2}

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        
        try:
            # Load audio file, resample to 16kHz as required by Whisper
            waveform, sr = librosa.load(item['audio_path'], sr=16000, mono=True)
        except Exception as e:
            # Simple error handling: if a file is corrupted, skip it and load the next one
            print(f"Error loading {item['audio_path']}: {e}. Skipping.")
            return self.__getitem__((idx + 1) % len(self))
            
        # Process the waveform to get input features for the model
        features = self.processor(waveform, sampling_rate=16000, return_tensors="pt").input_features
        
        # Convert hearing loss string to a numeric tensor
        hearing_loss_label = torch.tensor(self.hearing_loss_map.get(item['hearing_loss'], 0), dtype=torch.long)
        # Target value for regression
        correctness_score = torch.tensor([item['correctness']], dtype=torch.float32)

        # Return a dictionary of tensors and metadata (signal ID)
        return {
            "signal": item['signal'],
            "input_features": features.squeeze(0),
            "hearing_loss_label": hearing_loss_label,
            "correctness_score": correctness_score
        }

In [ ]:
# Define the Whisper Regression Model

import torch.nn as nn
from transformers import WhisperProcessor, WhisperModel

class WhisperRegressionModel(nn.Module):
    """
    A regression model that uses a pre-trained Whisper encoder.
    The Whisper backbone is frozen, and only a small regression head is trained.
    """
    def __init__(self, model_name="openai/whisper-large-v3", processor_name="openai/whisper-large-v3"):
        super().__init__()
        
        self.processor = WhisperProcessor.from_pretrained(processor_name)
        # Load the Whisper model with SDPA for better performance
        self.whisper = WhisperModel.from_pretrained(model_name, attn_implementation="sdpa")
        
        # Freeze the entire Whisper model
        self.whisper.requires_grad_(False)
        self.whisper.eval()

        hidden_size = self.whisper.config.d_model
        embedding_dim = 32

        # An embedding layer for the hearing loss categorical feature
        self.hearing_loss_embedding = nn.Embedding(num_embeddings=3, embedding_dim=embedding_dim)

        # The regression head that takes the combined features and outputs a score
        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size + embedding_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.5), # Increased dropout for regularization
            nn.Linear(512, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid() # Sigmoid to constrain output between 0 and 1
        )

    def forward(self, input_features, hearing_loss_label):
        # Get encoder hidden states from the frozen Whisper model
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            hidden_states = self.whisper.encoder(input_features).last_hidden_state
        
        # Pool the hidden states (average over the time dimension)
        pooled_output = torch.mean(hidden_states, dim=1)
        # Get the hearing loss embedding
        hl_embedding = self.hearing_loss_embedding(hearing_loss_label)
        # Concatenate audio features and hearing loss embedding
        combined_features = torch.cat((pooled_output, hl_embedding), dim=1)
        
        # Pass the combined features through the regression head
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            output = self.regression_head(combined_features)
            
        return output

In [ ]:
# Configure and Set Up Training

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import gc

# Free up GPU memory
gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Training Configuration
MODEL_NAME = "openai/whisper-large-v3"
BATCH_SIZE = 8
ACCUMULATION_STEPS = 8  # Effective batch size will be BATCH_SIZE * ACCUMULATION_STEPS
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
EPOCHS = 10 

model = WhisperRegressionModel(model_name=MODEL_NAME).to(DEVICE)
processor = model.processor

# Create PyTorch datasets
train_dataset = CadenzaDataset(split_dataset["train"], processor)
valid_dataset = CadenzaDataset(split_dataset["validation"], processor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)

# Setup optimizer, loss function, scaler, and scheduler
criterion = nn.MSELoss()
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, verbose=True)

print(f"Setup complete. Training with effective batch size of {BATCH_SIZE * ACCUMULATION_STEPS}")

Using device: cuda
Setup complete. Training with effective batch size of 64
DataLoader workers set to 0 to ensure stability.


/tmp/ipykernel_48/3284206543.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
# Run the Training & Evaluation Loop (just with early stopping)
import numpy as np
from tqdm.auto import tqdm
import math

# Early Stopping Parameters
best_val_rmse = float('inf')
patience_counter = 0
MAX_PATIENCE = 3 # Stop if validation RMSE doesn't improve for 3 consecutive epochs

for epoch in range(EPOCHS):
    model.train()
    model.regression_head.train()
    model.hearing_loss_embedding.train()
    
    running_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]")
    optimizer.zero_grad()
    
    for i, batch in enumerate(train_pbar):
        input_features = batch['input_features'].to(DEVICE)
        hearing_loss_label = batch['hearing_loss_label'].to(DEVICE)
        correctness_score = batch['correctness_score'].to(DEVICE)

        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(input_features, hearing_loss_label)
            loss = criterion(outputs, correctness_score)
            loss = loss / ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUMULATION_STEPS
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        train_pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    valid_pbar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]")
    with torch.no_grad():
        for batch in valid_pbar:
            input_features = batch['input_features'].to(DEVICE)
            hearing_loss_label = batch['hearing_loss_label'].to(DEVICE)
            correctness_score = batch['correctness_score'].to(DEVICE)
            
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(input_features, hearing_loss_label)
                loss = criterion(outputs, correctness_score)
                
            val_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(correctness_score.cpu().numpy())

    avg_val_loss = val_loss / len(valid_loader)
    all_preds, all_labels = np.concatenate(all_preds), np.concatenate(all_labels)
    val_rmse = math.sqrt(np.mean((all_preds - all_labels)**2))

    print(f"\nEpoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f} | Val RMSE: {val_rmse:.4f}")

    # Early Stopping and Model Checkpointing
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        torch.save(model.state_dict(), "best_whisper_regression_model.pth")
        patience_counter = 0
        print(f"  ✓ New best model saved with validation RMSE: {best_val_rmse:.4f}")
    else:
        patience_counter += 1
        print(f"  ✗ No improvement (Patience: {patience_counter}/{MAX_PATIENCE})")
        if patience_counter >= MAX_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch+1} epochs.")
            break
            
    # Step the learning rate scheduler
    scheduler.step(val_rmse)

print("\nTraining complete!")
print(f"Best validation RMSE achieved: {best_val_rmse:.4f}")

Epoch 1/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

Epoch 1/10 [Validation]:   0%|          | 0/56 [00:00<?, ?it/s]


Epoch 1/10 | Train Loss: 0.1131 | Val MSE: 0.0745 | Val RMSE: 0.2733
  ✓ New best model saved with validation RMSE: 0.2733


Epoch 2/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

Epoch 2/10 [Validation]:   0%|          | 0/56 [00:00<?, ?it/s]

...

Epoch 5/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

Epoch 5/10 [Validation]:   0%|          | 0/56 [00:00<?, ?it/s]


Epoch 5/10 | Train Loss: 0.0839 | Val MSE: 0.0702 | Val RMSE: 0.2656
  ✗ No improvement (Patience: 1/3)


Epoch 6/10 [Training]:   0%|          | 0/991 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Generate Submission File

After training, we load the best-performing model checkpoint and use it to generate predictions on our validation set. The predictions are then formatted into the required `submission.csv` format.

Given that the generation of the submission file was done in another enviroment than the one the model was trained in, the following cells can be run independently from all the above code cells. Hence there may be some duplicate downloads and vice versa.

In [9]:
#  1. Full Cleanup of Working Directory 
print("Performing a full cleanup of the working directory...")
!rm -rf /kaggle/working/*
print(" Cleanup complete.")

#  2. Install System Dependencies 
print("\nInstalling system packages (aria2 for accelerated downloads)...")
!apt-get update -qq && apt-get install -y -qq aria2
print(" System packages installed.")

#  3. Install Python Libraries with Pinned Versions 
print("\nInstalling required Python packages with specific versions...")
!pip install -q "transformers==4.34.0" \
                 "accelerate==0.23.0" \
                 "huggingface-hub==0.17.3" \
                 "datasets" \
                 "torch" \
                 "pandas" \
                 "librosa" \
                 "soundfile" \
                 "protobuf==3.20.3" \
                 "sentencepiece" \
                 "tqdm"
print(" Python packages installed.")

Performing a full cleanup of the working directory...
 Cleanup complete.

Installing system packages (aria2 for accelerated downloads)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 System packages installed.

Installing required Python packages with specific versions...
 Python packages installed.


In [10]:
import os

#  Configuration 
WORKING_DIR = "/kaggle/working"
DATA_DIR = os.path.join(WORKING_DIR, "cadenza_data")

# URLs
TRAIN_URL = "https://zenodo.org/records/17252365/files/cadenza_clip1_data.train.v1.0.tar.gz?download=1"
VALID_URL = "https://zenodo.org/records/17252365/files/cadenza_clip1_data.valid.v1.0.tar.gz?download=1"
EVAL_URL = "https://zenodo.org/records/17476436/files/cadenza_clip1_data.eval.v1.0.tar.gz?download=1"

# Archive paths
TRAIN_ARCHIVE = "train.tar.gz"
VALID_ARCHIVE = "valid.tar.gz"
EVAL_ARCHIVE = "eval.tar.gz"

os.makedirs(DATA_DIR, exist_ok=True)

#  Download 
print("Downloading all datasets...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{TRAIN_URL}" -o "{TRAIN_ARCHIVE}"
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{VALID_URL}" -o "{VALID_ARCHIVE}"
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{EVAL_URL}" -o "{EVAL_ARCHIVE}"
print(" All datasets downloaded.")

#  Extract 
print("\nExtracting all data into a unified directory...")
!tar -xzf {TRAIN_ARCHIVE} --strip-components=1 -C {DATA_DIR}
!tar -xzf {VALID_ARCHIVE} --strip-components=1 -C {DATA_DIR}
!tar -xzf {EVAL_ARCHIVE} --strip-components=1 -C {DATA_DIR}
print(" Data extraction complete.")

#  Cleanup 
print("\nCleaning up downloaded archives...")
!rm {TRAIN_ARCHIVE} {VALID_ARCHIVE} {EVAL_ARCHIVE}
print(" Cleanup complete.")

#  Verification 
print("\nVerifying final data directory structure:")
!ls -l {DATA_DIR}

 *** Download Progress Summary as of Mon Nov 17 12:36:42 2025 ***              m7m34s]m
[#242cc7 501MiB/4.2GiB(11%) CN:16 DL:7.2MiB ETA:8m43s]
FILE: /kaggle/working/train.tar.gz
-

 *** Download Progress Summary as of Mon Nov 17 12:37:42 2025 ***              m8m31s]
[#242cc7 0.9GiB/4.2GiB(22%) CN:16 DL:7.6MiB ETA:7m15s]
FILE: /kaggle/working/train.tar.gz
-

 *** Download Progress Summary as of Mon Nov 17 12:38:42 2025 ***              m6m14s]
[#242cc7 1.4GiB/4.2GiB(34%) CN:16 DL:8.3MiB ETA:5m40s]
FILE: /kaggle/working/train.tar.gz
-

 *** Download Progress Summary as of Mon Nov 17 12:39:43 2025 ***              m4m46s]
[#242cc7 1.8GiB/4.2GiB(45%) CN:16 DL:8.4MiB ETA:4m39s]
FILE: /kaggle/working/train.tar.gz
-

 *** Download Progress Summary as of Mon Nov 17 12:40:44 2025 ***              m4m15s]
[#242cc7 2.3GiB/4.2GiB(56%) CN:16 DL:7.3MiB ETA:4m17s]
FILE: /kaggle/working/train.tar.gz
-

 *** Download Progress Summary as of Mon Nov 17 12:41:45 2025 ***              m3m30s]
[#242cc7 2.8

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import WhisperProcessor, WhisperModel
import librosa

#  PyTorch Dataset Class 
class CadenzaDataset(Dataset):
    def __init__(self, hf_dataset, processor):
        self.hf_dataset = hf_dataset
        self.processor = processor
        self.hearing_loss_map = {"No Loss": 0, "Mild": 1, "Moderate": 2}

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        try:
            waveform, sr = librosa.load(item['audio_path'], sr=16000, mono=True)
        except Exception:
            # Skip corrupted files by loading the next one
            return self.__getitem__((idx + 1) % len(self))
            
        features = self.processor(waveform, sampling_rate=16000, return_tensors="pt").input_features
        hearing_loss_label = torch.tensor(self.hearing_loss_map.get(item['hearing_loss'], 0), dtype=torch.long)
        # Gracefully handle missing 'correctness' key for the eval set
        correctness_score = torch.tensor([item.get('correctness', 0.0)], dtype=torch.float32)

        return {
            "signal": item['signal'],
            "input_features": features.squeeze(0),
            "hearing_loss_label": hearing_loss_label,
            "correctness_score": correctness_score
        }

#  PyTorch Model Class 
class WhisperRegressionModel(nn.Module):
    def __init__(self, model_name="openai/whisper-large-v3"):
        super().__init__()
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.whisper = WhisperModel.from_pretrained(model_name)  
        self.whisper.requires_grad_(False)
        self.whisper.eval()
        hidden_size = self.whisper.config.d_model
        embedding_dim = 32
        self.hearing_loss_embedding = nn.Embedding(num_embeddings=3, embedding_dim=embedding_dim)
        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size + embedding_dim, 512), nn.GELU(), nn.LayerNorm(512),
            nn.Dropout(0.5), nn.Linear(512, 256), nn.GELU(), nn.LayerNorm(256),
            nn.Dropout(0.5), nn.Linear(256, 1), nn.Sigmoid()
        )

    def forward(self, input_features, hearing_loss_label):
        with torch.no_grad(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            hidden_states = self.whisper.encoder(input_features).last_hidden_state
        pooled_output = torch.mean(hidden_states, dim=1)
        hl_embedding = self.hearing_loss_embedding(hearing_loss_label)
        combined_features = torch.cat((pooled_output, hl_embedding), dim=1)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            output = self.regression_head(combined_features)
        return output

print(" Model and Dataset classes defined.")

 Model and Dataset classes defined.


In [18]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np
import torch
import os
from torch.utils.data import DataLoader
from datasets import load_dataset

import gc
import torch

# Clear GPU memory before starting
gc.collect()
torch.cuda.empty_cache()

#  IMPORTANT: Set your official Team ID here 
TEAM_ID = "T063"

#  Configuration 
DATA_DIR = "/kaggle/working/cadenza_data"
MODEL_NAME = "openai/whisper-large-v3-turbo"
MODEL_PATH = "/kaggle/input/best-whisper-regression-model-large-turbo/pytorch/default/1/best_whisper_regression_model.pth"
BATCH_SIZE = 2 # batch of size 2 because of out of memory issues.
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  1. Load Your Trained Model 
print(f"Loading your trained model from {MODEL_PATH}...")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"CRITICAL: Model file not found at {MODEL_PATH}. Make sure it is uploaded.")
inference_model = WhisperRegressionModel(model_name=MODEL_NAME).to(DEVICE)
inference_model.load_state_dict(torch.load(MODEL_PATH))
inference_model.eval()
print(" Model loaded successfully.")

#  2. Prepare DataLoaders 
print("\nPreparing DataLoaders...")
processor = inference_model.processor

# -- Validation DataLoader --
valid_metadata_path = os.path.join(DATA_DIR, "metadata/valid_metadata.json")
valid_audio_base_path = os.path.join(DATA_DIR, "valid/signals")
valid_hf_dataset = load_dataset('json', data_files=valid_metadata_path, split='train')
def add_valid_path(ex):
    ex['audio_path'] = os.path.join(valid_audio_base_path, f"{ex['signal']}.flac")
    return ex
valid_hf_dataset = valid_hf_dataset.map(add_valid_path)
valid_pytorch_dataset = CadenzaDataset(valid_hf_dataset, processor)
valid_loader = DataLoader(valid_pytorch_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# -- Evaluation DataLoader --
eval_metadata_path = os.path.join(DATA_DIR, "metadata/eval_metadata.json")
eval_audio_base_path = os.path.join(DATA_DIR, "eval/signals")
eval_hf_dataset = load_dataset('json', data_files=eval_metadata_path, split='train')
def add_eval_path(ex):
    ex['audio_path'] = os.path.join(eval_audio_base_path, f"{ex['signal']}.flac")
    return ex
eval_hf_dataset = eval_hf_dataset.map(add_eval_path)
eval_pytorch_dataset = CadenzaDataset(eval_hf_dataset, processor)
eval_loader = DataLoader(eval_pytorch_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(" DataLoaders ready.")

#  3. Run Inference & Create Submission Files 
def run_inference(dataloader, model, device, desc):
    all_signal_ids, all_predictions = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_features = batch['input_features'].to(device)
            hearing_loss_label = batch['hearing_loss_label'].to(device)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(input_features, hearing_loss_label)
            all_predictions.extend(outputs.cpu().numpy().flatten() * 100)
            all_signal_ids.extend(batch['signal'])
    return pd.DataFrame({'signal_id': all_signal_ids, 'intelligibility_score': all_predictions})

valid_submission_df = run_inference(valid_loader, inference_model, DEVICE, "Validation Set Inference")
eval_submission_df = run_inference(eval_loader, inference_model, DEVICE, "Evaluation Set Inference")

# Save to correctly named CSV files
valid_submission_filename = f"ICASSP2026_valid_{TEAM_ID}.csv"
eval_submission_filename = f"ICASSP2026_eval_{TEAM_ID}.csv"
valid_submission_df.to_csv(valid_submission_filename, index=False)
eval_submission_df.to_csv(eval_submission_filename, index=False)

print(f"\n Final submission files created in /kaggle/working/:")
print(f" - {valid_submission_filename}")
print(f" - {eval_submission_filename}")

#  Final Verification 
print("\n Evaluation Submission Preview ")
!head {eval_submission_filename}

Loading your trained model from /kaggle/input/best-whisper-regression-model-large-turbo/pytorch/default/1/best_whisper_regression_model.pth...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


 Model loaded successfully.

Preparing DataLoaders...
 DataLoaders ready.


Validation Set Inference:   0%|          | 0/588 [00:00<?, ?it/s]

2025-11-17 12:58:32.853211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763384312.898670    3104 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763384312.911113    3104 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-17 12:58:32.951406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763384312.990784    3103 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763384313.002980    3103 cuda_blas.cc:1

Evaluation Set Inference:   0%|          | 0/548 [00:00<?, ?it/s]

2025-11-17 13:08:23.253515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-17 13:08:23.255126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763384903.298007    3161 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763384903.299418    3162 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763384903.312138    3161 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1763384903.313342    3162 cuda_blas.cc:1


 Final submission files created in /kaggle/working/:
 - ICASSP2026_valid_T063.csv
 - ICASSP2026_eval_T063.csv

 Evaluation Submission Preview 
signal_id,intelligibility_score
7beca240297136ec3376a8bf,33.2
c69cbeb052836d03da94a0e2,26.73
d4aa1bf28c6c54d126ec10d3,58.2
b74eff8d257368ad6baec225,27.83
82464f5bcb8789cacf18fefd,33.53
19871202e5c811ccf2bbe1d5,48.72
57268951c1e8b11037478659,59.66
fdf2de516a42c4f3580db1e7,24.97
bdf6362ba1d381d09864dc46,75.75
